# SQL Fundamentals

## What is a Relational Database?

A **relational database** organizes data into tables (relations) with rows and columns, linked by keys.

| Concept | Description |
|---------|-------------|
| **Table / Relation** | Collection of rows with same structure |
| **Row / Tuple** | A single record |
| **Column / Attribute** | A field in the table |
| **Primary Key** | Unique identifier for each row |
| **Foreign Key** | Reference to primary key in another table |
| **Index** | Data structure for fast lookups |

## SQL Categories

| Category | Commands | Purpose |
|----------|----------|---------|
| DDL | CREATE, ALTER, DROP, TRUNCATE | Define structure |
| DML | INSERT, UPDATE, DELETE | Modify data |
| DQL | SELECT | Query data |
| DCL | GRANT, REVOKE | Permissions |
| TCL | BEGIN, COMMIT, ROLLBACK | Transactions |

In [1]:
import sqlite3
import pandas as pd

# Connect to SQLite (in-memory for demo)
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# --- DDL: CREATE TABLE ---
cursor.executescript("""
CREATE TABLE users (
    id       INTEGER PRIMARY KEY AUTOINCREMENT,
    username TEXT    NOT NULL UNIQUE,
    email    TEXT    NOT NULL UNIQUE,
    age      INTEGER CHECK (age >= 0 AND age <= 150),
    role     TEXT    DEFAULT 'user' CHECK (role IN ('admin','user','guest')),
    created_at DATETIME DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE products (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    name        TEXT    NOT NULL,
    price       REAL    NOT NULL CHECK (price > 0),
    category    TEXT,
    stock       INTEGER DEFAULT 0
);

CREATE TABLE orders (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id     INTEGER NOT NULL REFERENCES users(id),
    product_id  INTEGER NOT NULL REFERENCES products(id),
    quantity    INTEGER DEFAULT 1,
    total_price REAL,
    order_date  DATETIME DEFAULT CURRENT_TIMESTAMP
);
""")

# --- DML: INSERT ---
users = [
    ("alice", "alice@example.com", 30, "admin"),
    ("bob", "bob@example.com", 25, "user"),
    ("carol", "carol@example.com", 35, "user"),
    ("dave", "dave@example.com", 28, "user"),
    ("eve", "eve@example.com", 22, "guest"),
]
cursor.executemany("INSERT INTO users (username, email, age, role) VALUES (?, ?, ?, ?)", users)

products = [
    ("Laptop", 999.99, "Electronics", 50),
    ("Mouse", 29.99, "Electronics", 200),
    ("Keyboard", 79.99, "Electronics", 150),
    ("Desk", 299.99, "Furniture", 30),
    ("Chair", 199.99, "Furniture", 40),
]
cursor.executemany("INSERT INTO products (name, price, category, stock) VALUES (?, ?, ?, ?)", products)

orders = [
    (1, 1, 1, 999.99),
    (2, 2, 2, 59.98),
    (2, 3, 1, 79.99),
    (3, 1, 1, 999.99),
    (4, 4, 1, 299.99),
]
cursor.executemany("INSERT INTO orders (user_id, product_id, quantity, total_price) VALUES (?, ?, ?, ?)", orders)

conn.commit()
print("Tables created and data inserted")
print(pd.read_sql("SELECT * FROM users", conn))

Tables created and data inserted
   id username              email  age   role           created_at
0   1    alice  alice@example.com   30  admin  2026-06-19 11:25:55
1   2      bob    bob@example.com   25   user  2026-06-19 11:25:55
2   3    carol  carol@example.com   35   user  2026-06-19 11:25:55
3   4     dave   dave@example.com   28   user  2026-06-19 11:25:55
4   5      eve    eve@example.com   22  guest  2026-06-19 11:25:55


## SELECT Querying Data

In [2]:
# Basic SELECT
print("=== Basic SELECT ===")
print(pd.read_sql("SELECT username, email, age FROM users WHERE age > 25", conn))

# ORDER BY
print("\n=== ORDER BY ===")
print(pd.read_sql("SELECT username, age FROM users ORDER BY age DESC LIMIT 3", conn))

# LIKE pattern matching
print("\n=== LIKE ===")
print(pd.read_sql("SELECT username FROM users WHERE email LIKE '%example.com'", conn))

# IN operator
print("\n=== IN ===")
print(pd.read_sql("SELECT username, role FROM users WHERE role IN ('admin', 'guest')", conn))

# BETWEEN
print("\n=== BETWEEN ===")
print(pd.read_sql("SELECT username, age FROM users WHERE age BETWEEN 25 AND 30", conn))

=== Basic SELECT ===
  username              email  age
0    alice  alice@example.com   30
1    carol  carol@example.com   35
2     dave   dave@example.com   28

=== ORDER BY ===
  username  age
0    carol   35
1    alice   30
2     dave   28

=== LIKE ===
  username
0    alice
1      bob
2    carol
3     dave
4      eve

=== IN ===
  username   role
0    alice  admin
1      eve  guest

=== BETWEEN ===
  username  age
0    alice   30
1      bob   25
2     dave   28


## JOINs

```
INNER JOIN     LEFT JOIN      RIGHT JOIN      FULL OUTER JOIN
  A ∩ B          A ∪ (A∩B)    B ∪ (A∩B)        A ∪ B
  
  [A][A∩B][B]   [A][A∩B][  ]  [ ][A∩B][B]    [A][A∩B][B]
```

In [3]:
# INNER JOIN only matching rows
print("=== INNER JOIN ===")
q = """
SELECT u.username, p.name AS product, o.quantity, o.total_price
FROM orders o
INNER JOIN users u ON o.user_id = u.id
INNER JOIN products p ON o.product_id = p.id
"""
print(pd.read_sql(q, conn))

# LEFT JOIN all users, even those with no orders
print("\n=== LEFT JOIN (all users) ===")
q = """
SELECT u.username, COUNT(o.id) AS order_count
FROM users u
LEFT JOIN orders o ON u.id = o.user_id
GROUP BY u.id, u.username
ORDER BY order_count DESC
"""
print(pd.read_sql(q, conn))

# SELF JOIN compare rows in same table
print("\n=== SELF JOIN (users older than bob) ===")
q = """
SELECT a.username, a.age, b.username AS compared_to, b.age AS ref_age
FROM users a
JOIN users b ON b.username = 'bob'
WHERE a.age > b.age
"""
print(pd.read_sql(q, conn))

=== INNER JOIN ===
  username   product  quantity  total_price
0    alice    Laptop         1       999.99
1      bob     Mouse         2        59.98
2      bob  Keyboard         1        79.99
3    carol    Laptop         1       999.99
4     dave      Desk         1       299.99

=== LEFT JOIN (all users) ===
  username  order_count
0      bob            2
1    alice            1
2    carol            1
3     dave            1
4      eve            0

=== SELF JOIN (users older than bob) ===
  username  age compared_to  ref_age
0    alice   30         bob       25
1    carol   35         bob       25
2     dave   28         bob       25


## Aggregations & GROUP BY

In [4]:
# GROUP BY + Aggregations
q = """
SELECT 
    category,
    COUNT(*) AS product_count,
    AVG(price) AS avg_price,
    MIN(price) AS min_price,
    MAX(price) AS max_price,
    SUM(stock) AS total_stock
FROM products
GROUP BY category
HAVING COUNT(*) > 1
ORDER BY avg_price DESC
"""
print("=== GROUP BY + HAVING ===")
print(pd.read_sql(q, conn))

# Revenue per user
q = """
SELECT u.username, 
       COUNT(o.id) AS orders,
       SUM(o.total_price) AS total_spent,
       AVG(o.total_price) AS avg_order_value
FROM users u
JOIN orders o ON u.id = o.user_id
GROUP BY u.id, u.username
ORDER BY total_spent DESC
"""
print("\n=== Revenue per User ===")
print(pd.read_sql(q, conn))

=== GROUP BY + HAVING ===
      category  product_count  avg_price  min_price  max_price  total_stock
0  Electronics              3     369.99      29.99     999.99          400
1    Furniture              2     249.99     199.99     299.99           70

=== Revenue per User ===
  username  orders  total_spent  avg_order_value
0    alice       1       999.99          999.990
1    carol       1       999.99          999.990
2     dave       1       299.99          299.990
3      bob       2       139.97           69.985


## Subqueries & CTEs

In [5]:
# Subquery users who spent above average
q = """
SELECT u.username, SUM(o.total_price) AS total
FROM users u JOIN orders o ON u.id = o.user_id
GROUP BY u.id
HAVING total > (SELECT AVG(total_price) FROM orders)
"""
print("=== Subquery ===")
print(pd.read_sql(q, conn))

# CTE (WITH clause) cleaner version
q = """
WITH user_totals AS (
    SELECT u.username, SUM(o.total_price) AS total_spent
    FROM users u JOIN orders o ON u.id = o.user_id
    GROUP BY u.id, u.username
),
avg_spend AS (
    SELECT AVG(total_spent) AS avg FROM user_totals
)
SELECT ut.username, ut.total_spent, avg_spend.avg
FROM user_totals ut, avg_spend
WHERE ut.total_spent > avg_spend.avg
"""
print("\n=== CTE ===")
print(pd.read_sql(q, conn))

=== Subquery ===
  username   total
0    alice  999.99
1    carol  999.99

=== CTE ===
  username  total_spent      avg
0    alice       999.99  609.985
1    carol       999.99  609.985


## Window Functions

In [6]:
# Window functions (SQLite 3.25+)
# ROW_NUMBER, RANK, DENSE_RANK, LAG, LEAD

q = """
SELECT 
    name, category, price,
    ROW_NUMBER() OVER (PARTITION BY category ORDER BY price DESC) AS row_num,
    RANK()       OVER (PARTITION BY category ORDER BY price DESC) AS rank,
    DENSE_RANK() OVER (PARTITION BY category ORDER BY price DESC) AS dense_rank,
    AVG(price)   OVER (PARTITION BY category) AS category_avg_price,
    price - LAG(price, 1, price) OVER (PARTITION BY category ORDER BY price DESC) AS price_diff
FROM products
ORDER BY category, price DESC
"""
print("=== Window Functions ===")
print(pd.read_sql(q, conn))

=== Window Functions ===


       name     category   price  row_num  rank  dense_rank  \
0    Laptop  Electronics  999.99        1     1           1   
1  Keyboard  Electronics   79.99        2     2           2   
2     Mouse  Electronics   29.99        3     3           3   
3      Desk    Furniture  299.99        1     1           1   
4     Chair    Furniture  199.99        2     2           2   

   category_avg_price  price_diff  
0              369.99         0.0  
1              369.99      -920.0  
2              369.99       -50.0  
3              249.99         0.0  
4              249.99      -100.0  

## ACID Transactions

| Property | Meaning |
|----------|---------|
| **Atomicity** | All operations succeed or all fail (no partial updates) |
| **Consistency** | DB moves from one valid state to another |
| **Isolation** | Concurrent transactions don't interfere |
| **Durability** | Committed changes survive crashes |

### Isolation Levels (least → most strict)

| Level | Dirty Read | Non-repeatable Read | Phantom Read |
|-------|-----------|--------------------|--------------|
| READ UNCOMMITTED | ✅ possible | ✅ possible | ✅ possible |
| READ COMMITTED | ❌ prevented | ✅ possible | ✅ possible |
| REPEATABLE READ | ❌ | ❌ prevented | ✅ possible |
| SERIALIZABLE | ❌ | ❌ | ❌ prevented |

In [7]:
# Transaction example transfer money (atomicity)
def transfer_funds(conn, from_user, to_user, amount):
    cursor = conn.cursor()
    try:
        conn.execute("BEGIN")
        # Deduct from sender
        cursor.execute("UPDATE users SET age = age - ? WHERE id = ?", (amount, from_user))
        if cursor.rowcount == 0:
            raise Exception("Sender not found")
        # Add to receiver
        cursor.execute("UPDATE users SET age = age + ? WHERE id = ?", (amount, to_user))
        if cursor.rowcount == 0:
            raise Exception("Receiver not found")
        conn.execute("COMMIT")
        print(f"Transfer successful")
    except Exception as e:
        conn.execute("ROLLBACK")  # Undo all changes
        print(f"Transfer failed, rolled back: {e}")

# Using context manager (preferred)
with conn:
    conn.execute("UPDATE products SET stock = stock - 1 WHERE id = 1")
    # If anything raises here, it auto-rollbacks
print("Transaction committed")

Transaction committed


## Indexes & Performance

In [8]:
# Create indexes
cursor.execute("CREATE INDEX idx_users_email ON users(email)")
cursor.execute("CREATE INDEX idx_orders_user_id ON orders(user_id)")
cursor.execute("CREATE INDEX idx_products_category_price ON products(category, price)")
cursor.execute("CREATE UNIQUE INDEX idx_users_username ON users(username)")

# EXPLAIN QUERY PLAN (SQLite equivalent)
plan = cursor.execute("EXPLAIN QUERY PLAN SELECT * FROM users WHERE email = 'alice@example.com'").fetchall()
print("Query plan:", plan)

# Index types in PostgreSQL:
# B-tree: default, good for =, <, >, BETWEEN, LIKE 'prefix%'
# Hash:   only for = comparisons, faster but not on disk
# GiST:   geometric types, full-text search
# GIN:    arrays, JSONB, full-text search (tsvector)
# BRIN:   large tables with sequential data (timestamps)

print("\nPostgreSQL index creation examples (syntax):")
examples = [
    "CREATE INDEX idx_btree ON users(created_at);         , B-tree (default)",
    "CREATE INDEX idx_hash ON users USING HASH (email);  , Hash",
    "CREATE INDEX idx_gin ON docs USING GIN (content);   , Full-text GIN",
    "CREATE INDEX idx_partial ON orders(user_id) WHERE status='active'; , Partial index",
]
for ex in examples:
    print(" ", ex)

Query plan: [(3, 0, 0, 'SEARCH users USING INDEX sqlite_autoindex_users_2 (email=?)')]

PostgreSQL index creation examples (syntax):
  CREATE INDEX idx_btree ON users(created_at);         , B-tree (default)
  CREATE INDEX idx_hash ON users USING HASH (email);  , Hash
  CREATE INDEX idx_gin ON docs USING GIN (content);   , Full-text GIN
  CREATE INDEX idx_partial ON orders(user_id) WHERE status='active'; , Partial index


## Python + SQLAlchemy (ORM)

In [9]:
from sqlalchemy import create_engine, text
from sqlalchemy.orm import Session

engine = create_engine("sqlite:///:memory:", echo=False)

# Raw SQL with SQLAlchemy
with engine.connect() as connection:
    connection.execute(text("""
        CREATE TABLE IF NOT EXISTS demo (
            id INTEGER PRIMARY KEY,
            name TEXT,
            value REAL
        )
    """))
    connection.execute(text("INSERT INTO demo VALUES (1, 'test', 42.5)"))
    connection.commit()
    result = connection.execute(text("SELECT * FROM demo"))
    for row in result:
        print(row)

print("\nSQLAlchemy connection working")

(1, 'test', 42.5)

SQLAlchemy connection working


## Additional Learning Resources

### Official Documentation
- [PostgreSQL Docs](https://www.postgresql.org/docs/) Most complete SQL reference
- [SQLite Docs](https://www.sqlite.org/docs.html) Lightweight embedded DB
- [MySQL Docs](https://dev.mysql.com/doc/) Popular web DB
- [SQLAlchemy Docs](https://docs.sqlalchemy.org/en/20/) Python ORM

### Books
- [Learning SQL](https://www.oreilly.com/library/view/learning-sql-3rd/9781492057604/) Alan Beaulieu (O'Reilly)
- [SQL Cookbook](https://www.oreilly.com/library/view/sql-cookbook-2nd/9781492077435/) Anthony Molinaro
- [Designing Data-Intensive Applications](https://dataintensive.net/) Martin Kleppmann

### Interactive Practice
- [SQLZoo](https://sqlzoo.net/) Interactive SQL exercises
- [LeetCode SQL](https://leetcode.com/problemset/database/) SQL interview problems
- [Mode SQL Tutorial](https://mode.com/sql-tutorial/) Analytics SQL
- [pgexercises](https://pgexercises.com/) PostgreSQL exercises

### Videos
- [SQL Full Course - freeCodeCamp](https://www.youtube.com/watch?v=HXV3zeQKqGY) 4-hour full course
- [PostgreSQL Tutorial](https://www.youtube.com/watch?v=qw--VYLpxG4) Comprehensive guide